## Install PySpark and create a session

In [104]:
!pip install pyspark==4.1.2

In [105]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrameReader

#create a spark session
spark=SparkSession.builder.appName("AirQuality").getOrCreate()
spark

## Read in data to pyspark dataframe

In [106]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [107]:
#Read in previously saved Parquet File
df_aq=spark.read.parquet("/content/drive/MyDrive/AirQualityModel/processed.pq")
df_aq.show(5)

+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+
|            SiteName|      Date|        SiteType|Hour|ParticulateReading|         WindSpeed|     WindDirection|Month|DayOfWeek|
+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+
|Peterborough Gart...|2025-01-01|Urban Background|   1|            12.618|13.699999809265137|             236.5|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   2|             3.325|14.199999809265137|             236.0|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   3|             2.453|14.199999809265137|235.89999389648438|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   4|             1.934|12.699999809265137|235.89999389648438|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   5|             1.722|12.600000381469727|234.

In [108]:
df_aq.describe().show()

+-------+--------------+----------------+-----------------+------------------+------------------+------------------+-----+---------+
|summary|      SiteName|        SiteType|             Hour|ParticulateReading|         WindSpeed|     WindDirection|Month|DayOfWeek|
+-------+--------------+----------------+-----------------+------------------+------------------+------------------+-----+---------+
|  count|         96360|           96360|            96360|             96360|             96360|             96360|96360|    96360|
|   mean|          NULL|            NULL|             11.5| 8.247651266746475| 4.992274216778642|189.92013003571589| NULL|     NULL|
| stddev|          NULL|            NULL|6.922222471072395| 7.483436017919939|2.5183738008682073| 94.82765029884726| NULL|     NULL|
|    min|Barnstaple A39|Rural Background|                0|             0.283|               0.0|               0.0|  Apr|      Fri|
|    max|    Wicken Fen|   Urban Traffic|               23|          

#Pre-Processing for pipeline

identify categorical, numerical and label columns

In [109]:
#exclude date and site name from features to promote a more generic model
numerical = ['Hour','WindSpeed','WindDirection']
categorical = ['SiteType','Month','DayOfWeek']
categorical_indexed = [c + "_index" for c in categorical]
input_features = numerical + categorical_indexed
label = 'ParticulateReading'

#Create train and test data

As the distribution in the dataset is not even across all site types, we will startify the sampling by site type.

In [110]:
#create sub-lists on Urban Background, Rural Background and Urban Traffic

df_UB=df_aq.filter(df_aq.SiteType=='Urban Background')
df_RB=df_aq.filter(df_aq.SiteType=='Rural Background')
df_UT=df_aq.filter(df_aq.SiteType=='Urban Traffic')

#Split into train and test using reandom split
ubTrain,ubTest = df_UB.randomSplit(weights=[0.85,0.15],seed=42)
rbTrain,rbTest = df_RB.randomSplit(weights=[0.85,0.15],seed=42)
utTrain,utTest = df_UT.randomSplit(weights=[0.85,0.15],seed=42)

#recombine into overall test and train
df_train=ubTrain.union(rbTrain).union(utTrain)
df_test=ubTest.union(rbTest).union(utTest)

#count rows to check split
print(df_aq.count())
print(df_train.count())
print(df_test.count())

96360
82078
14282


# Create pipelines to test different machine learning regressor models (Decision Tree, Linear,random forest)


*   Index strings
*   vectorise features
*   Convert numericals to Scalars
*   ML algorithm




In [111]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

ind = StringIndexer(inputCols=categorical,outputCols=categorical_indexed,handleInvalid='skip')
vec = VectorAssembler(inputCols=input_features,outputCol='features',handleInvalid='skip')
scal = StandardScaler(inputCol='features',outputCol='scaled_features')
lin = LinearRegression(featuresCol='scaled_features',labelCol='ParticulateReading')
dt = DecisionTreeRegressor(featuresCol='scaled_features',labelCol='ParticulateReading')
rf = RandomForestRegressor(featuresCol='scaled_features',labelCol='ParticulateReading')

pipeline_lin = Pipeline(stages=[ind,vec,scal,lin])
pipeline_dt = Pipeline(stages=[ind,vec,scal,dt])
pipeline_rf = Pipeline(stages=[ind,vec,scal,rf])

prep_only = Pipeline(stages=[ind,vec,scal])


#Test the models

In [112]:
#linear regression first
model_lin = pipeline_lin.fit(df_train)
pred_lin = model_lin.transform(df_test)
pred_lin.show(5)

+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+--------------+-----------+---------------+--------------------+--------------------+------------------+
|            SiteName|      Date|        SiteType|Hour|ParticulateReading|         WindSpeed|     WindDirection|Month|DayOfWeek|SiteType_index|Month_index|DayOfWeek_index|            features|     scaled_features|        prediction|
+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+--------------+-----------+---------------+--------------------+--------------------+------------------+
|Borehamwood Meado...|2025-01-01|Urban Background|   6|             3.585|              11.0| 230.8000030517578|  Jan|      Wed|           0.0|        1.0|            0.0|[6.0,11.0,230.800...|[0.86668795667809...|3.5013184988695016|
|Borehamwood Meado...|2025-01-01|Urban Background|   8|             

In [113]:
#decision tree second
model_dt = pipeline_dt.fit(df_train)
pred_dt = model_dt.transform(df_test)
pred_dt.show(5)

+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+--------------+-----------+---------------+--------------------+--------------------+------------------+
|            SiteName|      Date|        SiteType|Hour|ParticulateReading|         WindSpeed|     WindDirection|Month|DayOfWeek|SiteType_index|Month_index|DayOfWeek_index|            features|     scaled_features|        prediction|
+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+--------------+-----------+---------------+--------------------+--------------------+------------------+
|Borehamwood Meado...|2025-01-01|Urban Background|   6|             3.585|              11.0| 230.8000030517578|  Jan|      Wed|           0.0|        1.0|            0.0|[6.0,11.0,230.800...|[0.86668795667809...| 4.963061110019401|
|Borehamwood Meado...|2025-01-01|Urban Background|   8|             

In [114]:
#random forest last
model_rf = pipeline_rf.fit(df_train)
pred_rf = model_rf.transform(df_test)
pred_rf.show(5)

+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+--------------+-----------+---------------+--------------------+--------------------+-----------------+
|            SiteName|      Date|        SiteType|Hour|ParticulateReading|         WindSpeed|     WindDirection|Month|DayOfWeek|SiteType_index|Month_index|DayOfWeek_index|            features|     scaled_features|       prediction|
+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+--------------+-----------+---------------+--------------------+--------------------+-----------------+
|Borehamwood Meado...|2025-01-01|Urban Background|   6|             3.585|              11.0| 230.8000030517578|  Jan|      Wed|           0.0|        1.0|            0.0|[6.0,11.0,230.800...|[0.86668795667809...|5.763916036534885|
|Borehamwood Meado...|2025-01-01|Urban Background|   8|             2.26

#Compare models and select best one

We will use the regerssion evaluator function to compare the fit of the models

In [115]:
#Start with linear
lin_eval=RegressionEvaluator()
lin_eval.setPredictionCol('prediction')
lin_eval.setLabelCol('ParticulateReading')
lin_rsme = lin_eval.evaluate(pred_lin)
lin_r2 = lin_eval.setMetricName('r2').evaluate(pred_lin)
lin_mae = lin_eval.setMetricName('mae').evaluate(pred_lin)

print('Linear RMSE:',lin_rsme)
print('Linear R2:',lin_r2)
print('Linear MAE:',lin_mae)


Linear RMSE: 7.162826661558575
Linear R2: 0.08164828725120787
Linear MAE: 4.761044763155709


In [116]:
#Repeat for Decision Tree
dt_eval=RegressionEvaluator()
dt_eval.setPredictionCol('prediction')
dt_eval.setLabelCol('ParticulateReading')
dt_rsme = dt_eval.evaluate(pred_dt)
dt_r2 = dt_eval.setMetricName('r2').evaluate(pred_dt)
dt_mae = dt_eval.setMetricName('mae').evaluate(pred_dt)

print('DT RMSE:',dt_rsme)
print('DT R2:',dt_r2)
print('DT MAE:',dt_mae)

DT RMSE: 6.2477990791308375
DT R2: 0.30129433112639337
DT MAE: 3.9814371172380394


In [117]:
#finally, random forest
rf_eval=RegressionEvaluator()
rf_eval.setPredictionCol('prediction')
rf_eval.setLabelCol('ParticulateReading')

rf_rsme = rf_eval.evaluate(pred_rf)
rf_r2 = rf_eval.setMetricName('r2').evaluate(pred_rf)
rf_mae = rf_eval.setMetricName('mae').evaluate(pred_rf)

print('RF RMSE:',rf_rsme)
print('RF R2:',rf_r2)
print('RF MAE:',rf_mae)

RF RMSE: 6.444893946373559
RF R2: 0.25651586132033355
RF MAE: 4.15895232296494


#Hyper parameter tuning of selected model using cross validation

From the above, we can see that the decision tree model is the best, although there is room for improvement, as the rmse is only slightly below the standard deviation, and the R2 suggests the model only accounts for 30% of variance in the target variable, so we will look at hyper-parameter tuning of the max depth, max bins and minimum instances per node.  We will use the rsme as the tuning evaluator as this gives a good allround view of the model accuracy.

In [118]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
#build the parameter grid
paramGrid = ParamGridBuilder()\
  .addGrid(dt.maxDepth,[2,5,10])\
  .addGrid(dt.maxBins,[16,32,48])\
  .addGrid(dt.minInstancesPerNode,[1,2,3])\
  .build()

In [119]:
#set the evaluator
evaluator=RegressionEvaluator(predictionCol='prediction',labelCol='ParticulateReading',metricName='rmse')


In [120]:
#Apply crossvalidator
crossval = CrossValidator(estimator=pipeline_dt,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3,
                          seed=42)

In [121]:
cv_dt_model = crossval.fit(df_train)

In [122]:
#evaluate and inspect the best model
best=cv_dt_model.bestModel
pred_best = best.transform(df_test)
best_rmse = evaluator.evaluate(pred_best)
best_r2 = evaluator.setMetricName('r2').evaluate(pred_best)
best_mae = evaluator.setMetricName('mae').evaluate(pred_best)

print('Best RMSE:',best_rmse)
print('Best R2:',best_r2)
print('Best MAE:',best_mae)

Best RMSE: 5.173658238339635
Best R2: 0.520889564058844
Best MAE: 3.1671091603830166


We can see that this has improved the model significantly, with a RMSE of 5.2 compared to the intial 6.2, and an r2 score of 0.52, suggesting that this model accounts for 52% of variation in the target variable

Fitting the cross-validation took a while (~7 minutes); use of TrainValidationSplit would have been quicker, but uses less folds

In [126]:
#Lets look at the parameter settings of the finalised model
best_maxDepth = best.stages[-1].getMaxDepth()
best_maxBins = best.stages[-1].getMaxBins()
best_minInstancesPerNode = best.stages[-1].getMinInstancesPerNode()


print('Best max Depth:',best_maxDepth)
print('Best max Bins:',best_maxBins)
print('Best min Instances Per Node:',best_minInstancesPerNode)

Best max Depth: 10
Best max Bins: 16
Best min Instances Per Node: 2


# Save the pipeline and model

In [127]:
#save pipeline
best.write().overwrite().save('pipeline_dt_tuned')
#save model only
best.stages[-1].write().overwrite().save('dt_model_tuned')